In [ ]:
import torch
from torch import nn
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
from rt_whisper.models.boundary_word_filter import BoundaryWordFilter

In [ ]:
EXPONENT = 2
MODELS = {
    "test": {
        "path": "/workspaces/dev/test/optimize/all/model_weight/model-pt-96k_3s_96bu.pth",
        "overlap_length": 96_000,
        "chunk_length": 48_000,
        "boundary": 96_000,
    }
}

In [ ]:
models = {k: Path(v["path"]) for k, v in MODELS.items()}

if not all(m.exists() for m in models.values()):
    raise FileNotFoundError(f"One or more model files do not exist: {models}")

In [ ]:
b_models = {k: BoundaryWordFilter.load(v) for k, v in models.items()}

In [ ]:
def generate_data(
    size:int,
    length:int,
    boundary: int,
    dur:int = -1
):
    b = np.random.randint(0, length+1, 2 * size).reshape(size, 2)
    start, end = b.min(axis=1).astype(np.int32), b.max(axis=1).astype(np.int32)
    start = (start // 160) * 160
    end = (end // 160) * 160

    if dur > 0:
        mask = (start + dur) <= length
        end[mask] = start[mask] + dur
        start[~mask] = end[~mask] - dur

        neg = start < 0
        if np.any(neg):
            start[neg] = 0
            end[neg] = dur

    mid = (start + end) // 2

    ss = np.clip(start/ boundary, 0, 1)
    se = np.clip(end/ boundary, 0, 1)
    es = np.clip((length - start) / boundary, 0, 1)
    ee = np.clip((length - end) / boundary, 0, 1)
    sm = np.clip(mid / boundary, 0, 1)
    em = np.clip((length - mid) / boundary, 0, 1)

    x = np.stack([ss, se, es, ee, sm, em], axis=1).astype(np.float32)

    near = np.minimum(mid, length - mid)
    y = np.clip(near / boundary, 0, 1) ** EXPONENT
    y = y.astype(np.float32)

    return zip(x, y)

def plot_mid_vs_pred_and_target(
    model: nn.Module,
    X: torch.Tensor,
    Y: torch.Tensor,
    device: str | torch.device = "cpu"
):
    model.to(device)
    X = X.to(device)
    Y = Y.to(device)

    with torch.no_grad():
        # mid만 추출 (X의 세 번째 컬럼)
        mids = X[:, 4].cpu().numpy()

        # 모델 예측
        preds = model(X).cpu().numpy().flatten()
        Y = Y.cpu().numpy().flatten()

    # 산점도 그리기
    plt.figure(figsize=(8,5))
    plt.scatter(mids, Y, alpha=0.5, label="Target (Y)", color="blue")
    plt.scatter(mids, preds, alpha=0.5, label="Model Output", color="red")
    plt.xlabel("mid")
    plt.ylabel("value")
    plt.title("Mid vs Model Prediction & Target")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

In [ ]:
def plot_with_dur(model:nn.Module, datasets:dict, device: str|torch.device = "cpu"):
    for dur, data in datasets.items():
        print(f"========================= {dur} ===============================")
        X, Y = zip(*data)
        X = torch.tensor(X, dtype=torch.float32)
        Y = torch.tensor(Y, dtype=torch.float32).reshape(-1, 1)

        plot_mid_vs_pred_and_target(model, X, Y, device)

In [ ]:
for name, model in b_models.items():
    print(name)
    ol = MODELS[name]["overlap_length"]
    cl = MODELS[name]["chunk_length"]
    boundary = MODELS[name]["boundary"]

    datasets = {dur:[data for data in generate_data(1024, length=ol + cl, boundary=boundary, dur=dur)] for dur in range(0, 16000, 1600)}
    plot_with_dur(model, datasets, "cuda")